# <center>TensorRT 引擎构建</center>

在上一节中，我们讨论了将训练完成的模型从训练框架迁移到 TensorRT 进行构建部署的三种策略：

1. 利用框架自带的接口，如 [TF-TRT](https://docs.nvidia.com/deeplearning/frameworks/tf-trt-user-guide/index.html) 和 [Torch-TensorRT](https://pytorch.org/TensorRT/)。
2. 将模型转换为 ONNX 格式，然后再转换为 TensorRT 引擎后进行推理部署。
3. 直接使用 TensorRT 的原生 API 来构建网络进行推理部署。

&emsp;&emsp;在这三种策略中，<b><font color="red">将模型转换为 ONNX 格式后再转换为 TensorRT 引擎进行部署</font></b>是目前性价比最高的方案。[ONNX](https://onnx.ai/onnx/intro/concepts.html) 提供了一种从多种训练框架导出模型的方法，使这些模型能够与 TensorRT 运行时兼容。导出 ONNX 模型要求模型中的算子必须被 ONNX 支持<sup><a id="fnref1" href="https://onnx.ai/onnx/operators/index.html">1</a></sup>，并且需要为 TensorRT 不支持的算子提供插件实现<sup><a id="fnref2" href="https://github.com/NVIDIA/TensorRT/tree/main/plugin">2</a></sup>。

## <center>导出 PyTorch 模型到 ONNX</center>

<center>
    <img src="./course_data/images/pytorch_onnx.png" width=100%>
</center>

将 PyTorch 模型转换为 TensorRT 的一种方法是将 PyTorch 模型导出为 ONNX 格式，然后将其转换为 TensorRT 引擎。

<div class="alert alert-block alert-info">
    <b>截至 PyTorch 2.1 版本，ONNX 导出器有两个版本</b>
    <ul>
       <li><p>
           <code><span>torch.onnx.dynamo_export</span></code> 是基于 TorchDynamo 技术的最新的（仍处于测试阶段）导出器，随 PyTorch 2.0 一起发布。
       </p></li>
       <li><p>
           <code><span><a href="https://pytorch.org/docs/stable/onnx_torchscript.html#functions">torch.onnx.export</a></span></code> 基于 TorchScript 后端，自 PyTorch 1.2.0 起可用。
       </p></li>
    </ul>
</div>

**`torch.onnx.export` 核心参数：**

| 参数 | 说明 | 注意事项 |
|:------:|:------:|:----------:|
| `opset_version` | ONNX算子集版本 | <b><font color="red">≥11保证兼容性</font></b> |
| `input_names` | 输入节点命名 | 避免使用特殊字符 |
| `output_names` | 输出节点命名 | 避免使用特殊字符 |
| `dynamic_axes` | 指定动态维度 | <b><font color="red">仅batch维度动态，其他维度固定</font></b> |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torchvision

dummy_input = torch.randn(8, 3, 224, 224) # 假设模型期望的输入是一个批次大小为8，3个颜色通道，224x224像素的图像
model = torchvision.models.alexnet(pretrained=True)

# 导出静态模型
static_onnx_path = "./course_data/models/ALEX/alexnet_bs8.onnx"
torch.onnx.export(model, dummy_input, static_onnx_path, verbose=False)

# 导出动态Batch模型，并设置输入输出名称
dynamic_onnx_path = "./course_data/models/ALEX/alexnet_dynamic.onnx"
torch.onnx.export(
    model, dummy_input, dynamic_onnx_path,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
   verbose=False
)

In [ ]:
import netron
from IPython.display import IFrame, display, HTML

display(HTML("<style>div.output_scroll { height: 30em; }</style>"))
url, port = netron.start(static_onnx_path, verbosity=0, browse=False)
display(IFrame(f'http://{url}:{port}', width='100%', height='620px'))

In [ ]:
url, port = netron.start(dynamic_onnx_path, verbosity=0, browse=False)
display(IFrame(f'http://{url}:{port}', width='100%', height='640px'))

## <center>将 ONNX 转换为 TensorRT 引擎</center>

将 ONNX 模型转换为 TensorRT 引擎主要有三种方法：

1. 使用 [`trtexec`](https://github.com/NVIDIA/TensorRT/tree/main/samples/trtexec) 工具
2. 使用 TensorRT API
3. 使用 [NVIDIA Nsight Deep Learning Designer](https://developer.nvidia.com/nsight-dl-designer) **New**

在本课程中，将重点使用 `trtexec` 来将 ONNX 模型转换为 TensorRT 引擎。要查看所有可用选项及其说明，请运行 `trtexec --help` 命令。有关常用命令行标志的解释，请参见 [Commonly Used Command-line Flags](https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/index.html#trtexec-flags) 或 [17. TensorRT 的命令行程序](https://developer.nvidia.com/zh-cn/blog/tensorrt-trtexec-cn/)。

In [ ]:
!trtexec --help

要使用 `trtexec` 将之前导出 ONNX 静态模型转换为 TensorRT 引擎，可以运行以下命令，这将把 `alexnet_bs8.onnx` 转换为名为 `alexnet_bs8.engine` 的 TensorRT 引擎。

```bash
trtexec --onnx=./models/alexnet_bs8.onnx --saveEngine=./models/alexnet_bs8.engine
```

In [ ]:
from course_functions.command_runner import run_command

display(HTML("<style>div.output_scroll { height: 25em; }</style>"))
run_command("/usr/src/tensorrt/bin/trtexec --onnx=./course_data/models/ALEX/alexnet_bs8.onnx --saveEngine=./course_data/models/ALEX/alexnet_bs8.engine")   

若要将之前导出的 ONNX 动态模型转为 TensorRT 引擎，则需要使用 `--minShapes`、`--optShapes` 和 `--maxShapes` 标志来控制输入形状的范围，包括批量大小。

In [ ]:
command = (
    f"/usr/src/tensorrt/bin/trtexec --onnx={dynamic_onnx_path} --saveEngine={dynamic_onnx_path.replace('.onnx', '.engine')} "
    "--minShapes=input:1x3x224x224 --optShapes=input:4x3x224x224 --maxShapes=input:8x3x224x224"
)

run_command(command)    

## 【扩展】使用 trt-engine-explorer 对 TensorRT 引擎进行可视化

<center>
    <img src="./course_data/images/trex.png" width=60%>
</center>

[trt-engine-explorer](https://github.com/NVIDIA/TensorRT/tree/main/tools/experimental/trt-engine-explorer) 是 NVIDIA® TensorRT™ 开发的一款用于分析和可视化 TensorRT 引擎文件的 Python 包。具体的安装与使用方法可参考 [Installation](https://github.com/NVIDIA/TensorRT/tree/main/tools/experimental/trt-engine-explorer#installation) 与 [Tutorial Notebook](https://github.com/NVIDIA/TensorRT/tree/main/tools/experimental/trt-engine-explorer/notebooks)。

首先，设置模型路径和相关文件的路径：

In [ ]:
import os

onnx_path = static_onnx_path
model_path_prefix = os.path.splitext(onnx_path)[0]
engine_path = f"{model_path_prefix}.engine"
graph_json = f"{model_path_prefix}.graph.json"
profiling_json = f"{model_path_prefix}.profile.json"

然后，构建并运行 trtexec 命令来生成 TensorRT 引擎，并导出相关的分析文件：

In [ ]:
command = (
    f"/usr/src/tensorrt/bin/trtexec --onnx={onnx_path} --saveEngine={engine_path} "
#     " --fp16 --precisionConstraints=obey --layerPrecisions=/features/features.0/Conv:fp32 "
    "--separateProfileRun "
    "--profilingVerbosity=detailed " 
    f"--exportProfile={profiling_json} "
    f"--exportLayerInfo={graph_json}"
)

run_command(command, False)

接下来，使用 trex 库来创建 `EnginePlan` 并可视化：

In [ ]:
from trex import EnginePlan
from trex.report_card import report_card_draw_plan_graph_extended

plan = EnginePlan(graph_json, profiling_json)
report_card_draw_plan_graph_extended(plan, engine_path)

In [ ]:
from IPython.display import SVG

display(HTML("<style>div.output_scroll { height: 35em; }</style>"))
display(SVG((f"{engine_path}.svg")))

## <center>总结</center>

本节课程详细介绍了如何将 PyTorch 模型导出为 ONNX 格式，并使用 `trtexec` 工具将其转换为 TensorRT 引擎。此外，还介绍了如何通过 `trt-engine-explorer` 工具对 TensorRT 引擎进行可视化和分析。

通过本节的学习，您应掌握将 PyTorch 模型导出为 ONNX 格式、转换为 TensorRT 引擎，并使用工具进行引擎分析和性能优化的完整流程。